#  Data Cleaning — Tweets SAVIA

Ce notebook nettoie les tweets en supprimant :

- les `@mentions`
- les `#hashtags`
- les URLs
- les emojis
- la ponctuation
- les espaces multiples

Puis il :
- filtre les comptes officiels (Free, Iliad, Assistance Freebox, etc.)
- exporte un fichier `tweets_cleaned_<timestamp>.csv` dans `data/silver/`
- enregistre une entrée qualité dans `savia/quality/monitoring_quality_log.csv`
    

In [ ]:
import pandas as pd
import re, os
from datetime import datetime
    

In [ ]:
import pandas as pd
import re, os
from datetime import datetime

RAW_PATH = "../data/raw/free tweet export 2.csv"
timestamp = datetime.now().strftime("%Y-%m-%d_%Hh%M")
print(f"{timestamp}")
SILVER_PATH = f"../data/silver/tweets_cleaned_1.csv"
QUALITY_LOG_PATH  = "../quality/quality_log_tweets.csv"



def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"@[\w_]+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"["
                  u"\U0001F600-\U0001F64F"
                  u"\U0001F300-\U0001F5FF"
                  u"\U0001F680-\U0001F6FF"
                  u"\U0001F1E0-\U0001F1FF"
                  "]+", "", text, flags=re.UNICODE)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = pd.read_csv(RAW_PATH)
# 🧼 Application du nettoyage
df["clean_text"] = df["full_text"].apply(clean_text)
df["cleaned_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
     
# 🚫 Suppression des comptes officiels (Free, Iliad, Assistance, etc.)
EXCLUDED_ACCOUNTS = [
    "free", "free_1337", "free1337", "groupeiliad", "iliad",
    "free_officiel", "free_official", "groupe_iliad",
    "assistance freebox", "assistance_freebox"
]

# Normalisation des noms en minuscules sans accents (optionnel)
import unicodedata

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
def text_length(s):
    if pd.isna(s):
        return 0
    return len(s)
df["screen_name_clean"] = df["screen_name"].apply(normalize_text)
df["name_clean"] = df["name"].apply(normalize_text)
df["text_length"] = df["clean_text"].apply(text_length)

# Filtrage
df_clean = df[
    ~df["screen_name_clean"].isin(EXCLUDED_ACCOUNTS) &
    ~df["name_clean"].isin(EXCLUDED_ACCOUNTS)
].copy()

print(f"✅ {len(df) - len(df_clean)} comptes officiels exclus.")
df_clean[["screen_name", "name"]].drop_duplicates().head()
df_clean = df_clean[df_clean["text_length"] > 0]

os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)
df_clean[["id", "created_at", "screen_name", "full_text", "clean_text", "cleaned_at"]].to_csv(SILVER_PATH, index=False)
print(f"✅ Exporté vers : {SILVER_PATH}")
    

2025-09-26_14h01


In [46]:
# 🧽 Fonction de nettoyage
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"@[\w_]+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"["
                  u"\U0001F600-\U0001F64F"
                  u"\U0001F300-\U0001F5FF"
                  u"\U0001F680-\U0001F6FF"
                  u"\U0001F1E0-\U0001F1FF"
                  "]+", "", text, flags=re.UNICODE)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
    

In [47]:
# 📥 Chargement des données brutes
df = pd.read_csv(RAW_PATH)
df.head()
    

,id,created_at,full_text,media,screen_name,name,profile_image_url,user_id,in_reply_to,retweeted_status,...,favorite_count,retweet_count,bookmark_count,quote_count,reply_count,views_count,favorited,retweeted,bookmarked,url
0,1343458257915031553,2020-12-28 08:26:23 +01:00,"💩 à @free parce-que Débit Très instable, … \n\...",[],m_annuel,M Annuel,https://abs.twimg.com/sticky/default_profile_i...,1104790986801250304,NaN,NaN,...,2,1,0,0,1,NaN,False,False,False,https://twitter.com/m_annuel/status/1343458257...
1,1393158240083587075,2021-05-14 12:56:22 +02:00,"RT @free: Retrouvez désormais @ToonamiFR, la c...","[{""type"":""photo"",""url"":""https://t.co/kuAYafYDi...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.393125e+18,...,0,16,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/13931582400...
2,1403337211475546112,2021-06-11 15:03:58 +02:00,"RT @free: A suivre ce soir, le 1er match de l’...","[{""type"":""photo"",""url"":""https://t.co/gMTcYtGdd...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.403329e+18,...,0,15,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/14033372114...
3,1403337257571004417,2021-06-11 15:04:09 +02:00,RT @free: Disponible sur le canal 101 avec les...,[],Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,1.403333e+18,...,0,5,0,0,0,NaN,False,False,False,https://twitter.com/Freebox/status/14033372575...
4,1418550491034882052,2021-07-23 14:36:07 +02:00,« Faites vos premiers pas avec nous ! Découvre...,"[{""type"":""video"",""url"":""https://t.co/YCMv79evb...",Freebox,Assistance Freebox,https://pbs.twimg.com/profile_images/671676021...,58920430,NaN,NaN,...,31,7,0,3,35,NaN,False,False,False,https://twitter.com/Freebox/status/14185504910...


In [48]:
# 🧼 Application du nettoyage
df["clean_text"] = df["full_text"].apply(clean_text)
df["cleaned_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    

In [59]:
# 🚫 Suppression des comptes officiels (Free, Iliad, Assistance, etc.)
EXCLUDED_ACCOUNTS = [
    "free", "free_1337", "free1337", "groupeiliad", "iliad",
    "free_officiel", "free_official", "groupe_iliad",
    "assistance freebox", "assistance_freebox"
]

# Normalisation des noms en minuscules sans accents (optionnel)
import unicodedata

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
def text_length(s):
    if pd.isna(s):
        return 0
    return len(s)
df["screen_name_clean"] = df["screen_name"].apply(normalize_text)
df["name_clean"] = df["name"].apply(normalize_text)
df["text_length"] = df["clean_text"].apply(text_length)

# Filtrage
df_clean = df[
    ~df["screen_name_clean"].isin(EXCLUDED_ACCOUNTS) &
    ~df["name_clean"].isin(EXCLUDED_ACCOUNTS)
].copy()

print(f"✅ {len(df) - len(df_clean)} comptes officiels exclus.")
df_clean[["screen_name", "name"]].drop_duplicates().head()
df_clean = df_clean[df_clean["text_length"] > 0]

✅ 3324 comptes officiels exclus.


In [60]:
# Export du fichier nettoyé (filtré)
os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)
df_clean[["id", "created_at", "screen_name", "full_text", "clean_text", "cleaned_at"]].to_csv(SILVER_PATH, index=False)
print(f"✅ Exporté vers : {SILVER_PATH}")
    

✅ Exporté vers : ../data/silver/tweets_cleaned_1.csv


In [61]:
# Logging qualité enrichi

n_raw = len(df)
n_cleaned = len(df_clean)
n_delta = n_raw - n_cleaned
nulls = df_clean["full_text"].isna().sum()
pct_nulls = round(nulls / n_cleaned * 100, 2)
avg_length_cleaned = round(df_clean["clean_text"].str.len().mean(), 2)

log_entry = pd.DataFrame([{
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "raw_file": RAW_PATH,
    "export_file": SILVER_PATH,
    "n_total_raw": n_raw,
    "n_total_cleaned": n_cleaned,
    "n_removed_official": n_delta,
    "n_nulls": nulls,
    "pct_nulls": pct_nulls,
    "avg_length_clean_text": avg_length_cleaned
}])

os.makedirs(os.path.dirname(QUALITY_LOG_PATH), exist_ok=True)
if os.path.exists(QUALITY_LOG_PATH):
    print("📈 Ajout au log qualité existant")
    log_entry.to_csv(QUALITY_LOG_PATH, mode='a', header=False, index=False)
else:
    print("🆕 Création du log qualité else")
    log_entry.to_csv(QUALITY_LOG_PATH, index=False)


    

📈 Ajout au log qualité existant


In [2]:
import pandas as pd
df_log = pd.read_csv("/Users/eliot/Desktop/projet_rncp/SAVIA/data/silver/api_benchmark_2025-10-24_17h58.csv")

In [3]:
df_log

,id,model,status_code,start_time,end_time,duration_sec,response_text
0,1,Mistral-7B-Instruct,200,2025-10-24 17:58:10,2025-10-24 17:58:44,34.553,1. Je suis désolé pour apprendre que vous renc...
1,2,Mistral-medium,200,2025-10-24 17:58:10,2025-10-24 17:59:49,99.005,1. **Vérifie les câbles** :\n - Branchement ...
2,3,Mistral-7B-Instruct,200,2025-10-24 17:58:10,2025-10-24 17:58:17,7.214,1. Je suis désolé pour entendre que vous renco...
3,4,Mistral-medium,200,2025-10-24 17:58:10,2025-10-24 17:58:44,34.549,1. **Vérifie la cause du clignotement rouge** ...
4,5,Mistral-7B-Instruct,200,2025-10-24 17:58:10,2025-10-24 17:58:34,24.561,1. J'ai entendu que votre Box est en train de ...
...,...,...,...,...,...,...,...
995,996,Mistral-medium,ERROR: TimeoutError,2025-10-24 20:23:33,2025-10-24 20:26:04,150.981,Exception:
996,997,Mistral-7B-Instruct,ERROR: TimeoutError,2025-10-24 20:23:33,2025-10-24 20:26:04,150.981,Exception:
997,998,Mistral-medium,ERROR: TimeoutError,2025-10-24 20:23:33,2025-10-24 20:26:04,150.980,Exception:
998,999,Mistral-7B-Instruct,ERROR: TimeoutError,2025-10-24 20:23:33,2025-10-24 20:26:04,150.980,Exception:


In [7]:
df_log[df_log["status_code"] == "200"]

,id,model,status_code,start_time,end_time,duration_sec,response_text
0,1,Mistral-7B-Instruct,200,2025-10-24 17:58:10,2025-10-24 17:58:44,34.553,1. Je suis désolé pour apprendre que vous renc...
1,2,Mistral-medium,200,2025-10-24 17:58:10,2025-10-24 17:59:49,99.005,1. **Vérifie les câbles** :\n - Branchement ...
2,3,Mistral-7B-Instruct,200,2025-10-24 17:58:10,2025-10-24 17:58:17,7.214,1. Je suis désolé pour entendre que vous renco...
3,4,Mistral-medium,200,2025-10-24 17:58:10,2025-10-24 17:58:44,34.549,1. **Vérifie la cause du clignotement rouge** ...
4,5,Mistral-7B-Instruct,200,2025-10-24 17:58:10,2025-10-24 17:58:34,24.561,1. J'ai entendu que votre Box est en train de ...
...,...,...,...,...,...,...,...
432,433,Mistral-7B-Instruct,200,2025-10-24 19:00:39,2025-10-24 19:02:45,126.833,1. I'm here to help you with your Freebox issu...
436,437,Mistral-7B-Instruct,200,2025-10-24 19:00:39,2025-10-24 19:01:55,76.451,1. Je suis désolé pour entendre que vous renco...
494,495,Mistral-7B-Instruct,200,2025-10-24 19:08:12,2025-10-24 19:10:00,108.314,1. Je m'excuse pour ce problème incommodant.\n...
678,679,Mistral-7B-Instruct,200,2025-10-24 19:31:49,2025-10-24 19:40:14,137.914,1. Je suis désolé pour la gêne causée par ce p...
